## Cell 0 — Imports & Configuration
> Edit `GCS_BUCKET`, `FACT_URI`, `DIM_ART_URI` before running.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import io, os, re, json, hashlib, warnings, pathlib, datetime
warnings.filterwarnings("ignore")

# ── Numeric / ML ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.preprocessing import LabelEncoder

# ── GCS ───────────────────────────────────────────────────────────────────────
from google.cloud import storage as gcs_storage
import google.auth

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit the two URIs before running
# ══════════════════════════════════════════════════════════════════════════════
GCS_BUCKET   = "bucket-isdi-mda-online"
FACT_URI     = "gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/fact_lineas_albaran.csv"
DIM_ART_URI  = "gs://bucket-isdi-mda-online/proyecto-troncal/cruzber/dim_articulo.csv"

# Todos los outputs (caché + resultados) van al directorio del notebook
NB_DIR      = pathlib.Path(os.getcwd())
CACHE_DIR   = NB_DIR
OUT_DIR     = NB_DIR
REPORT_DIR  = NB_DIR

HORIZON_WEEKS    = 4
VAL_START        = pd.Timestamp("2024-01-01")
VAL_END          = pd.Timestamp("2024-12-31")
CALIB_WEEKS      = 26
ACTIVE_THRESHOLD = 5.0

print("Config OK")
print(f"  FACT   : {FACT_URI}")
print(f"  DIM    : {DIM_ART_URI}")
print(f"  NB_DIR : {NB_DIR}")
print(f"  VAL    : {VAL_START.date()} → {VAL_END.date()}")
print(f"  CALIB  : {CALIB_WEEKS} weeks before VAL_START")

## Cell 1 — GCS Download Helpers & Spanish Numeric Parser

In [ ]:
# ── Auth (ADC — Workbench default) ────────────────────────────────────────────
credentials, _proj = google.auth.default()
gcs_client = gcs_storage.Client(credentials=credentials)
print(f"GCS auth : {type(credentials).__name__}")


def parse_es_number(series: pd.Series) -> pd.Series:
    """Parse Spanish-formatted numbers: '1.234,56' -> 1234.56"""
    if pd.api.types.is_numeric_dtype(series):
        return series
    s = series.astype(str)
    s = s.str.replace(r"\.(?=\d{3})", "", regex=True)  # remove thousands dot
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def _uri_to_local(uri: str) -> pathlib.Path:
    slug = hashlib.md5(uri.encode()).hexdigest()[:12]
    return CACHE_DIR / f"{slug}.csv"


def download_from_gcs(gs_uri: str, local_path: pathlib.Path, force: bool = False) -> pathlib.Path:
    """Download gs://bucket/blob to local_path; skip if cached."""
    if local_path.exists() and not force:
        print(f"  [cache] {local_path.name}")
        return local_path
    uri = gs_uri.replace("gs://", "")
    bucket_name, blob_name = uri.split("/", 1)
    bucket = gcs_client.bucket(bucket_name)
    blob   = bucket.blob(blob_name)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(local_path))
    print(f"  [gcs]   gs://{bucket_name}/{blob_name}  -> {local_path.name}  ({local_path.stat().st_size:,} B)")
    return local_path


def load_df(uri: str) -> pd.DataFrame:
    """Download (cached) and load CSV into DataFrame."""
    local = _uri_to_local(uri)
    download_from_gcs(uri, local)
    return pd.read_csv(local, low_memory=False)


print("Helpers ready.")

## Cell 2 — Load Data (with GCS caching)
> Downloads once; subsequent runs use local cache.

In [ ]:
print("Loading fact_lineas_albaran …")
df_fact = load_df(FACT_URI)
print(f"  raw shape: {df_fact.shape}")
print(f"  columns:   {df_fact.columns.tolist()}")

# Load dim_articulo (optional — tolerate 404)
try:
    df_art = load_df(DIM_ART_URI)
    print(f"  dim_art:   {df_art.shape}")
    HAS_DIM = True
except Exception as e:
    print(f"  dim_art:   NOT LOADED ({e})")
    df_art  = pd.DataFrame()
    HAS_DIM = False

# ── Column name normalisation ─────────────────────────────────────────────────
df_fact.columns = df_fact.columns.str.strip().str.lower()
if HAS_DIM:
    df_art.columns = df_art.columns.str.strip().str.lower()

# Mandatory column aliases
alias = {
    "codigo_articulo": ["sku_id","article_id","articulo","cod_articulo","cod_art"],
    "fecha_albaran":   ["fecha","date","fecha_venta","invoice_date","fecha_factura"],
    "unidades":        ["qty","quantity","cantidad","units","unidades_vendidas"],
    "base_imponible":  ["importe","revenue","importe_ventas","base_imp","netsales"],
}
for canon, alts in alias.items():
    if canon not in df_fact.columns:
        for a in alts:
            if a in df_fact.columns:
                df_fact = df_fact.rename(columns={a: canon})
                print(f"  renamed '{a}' -> '{canon}'")
                break

# Hard assert mandatory columns
required = ["codigo_articulo","fecha_albaran","unidades","base_imponible"]
missing  = [c for c in required if c not in df_fact.columns]
assert not missing, f"FATAL — missing columns: {missing}. Available: {df_fact.columns.tolist()}"

# ── Parse dates ───────────────────────────────────────────────────────────────
df_fact["fecha_albaran"] = pd.to_datetime(df_fact["fecha_albaran"], errors="coerce", dayfirst=True)
nat_rate = df_fact["fecha_albaran"].isna().mean()
print(f"  fecha_albaran NaT rate: {nat_rate:.3%}")
df_fact = df_fact[df_fact["fecha_albaran"].notna()].copy()

# ── Parse numerics (Spanish format) ──────────────────────────────────────────
for col in ["unidades","base_imponible","importe_coste"]:
    if col in df_fact.columns:
        before = df_fact[col].isna().mean()
        df_fact[col] = parse_es_number(df_fact[col])
        after  = df_fact[col].isna().mean()
        print(f"  {col}: NaN {before:.3%} -> {after:.3%}  (non-null mean={df_fact[col].mean():.2f})")

HAS_COSTE = "importe_coste" in df_fact.columns and df_fact["importe_coste"].notna().sum() > 0

# Hard asserts
assert (df_fact["unidades"] > 0).sum() > 0, "FATAL — no positive unidades"
assert df_fact["unidades"].sum() > 0,        "FATAL — sum(unidades) == 0"
print(f"\nFact loaded: {len(df_fact):,} rows  |  SKUs: {df_fact['codigo_articulo'].nunique():,}  |  HAS_COSTE={HAS_COSTE}")

## Cell 3 — Build Weekly Panel (SKU × ISO-week)

In [ ]:
df_fact["sku_id"] = df_fact["codigo_articulo"].astype(str).str.strip()

# ISO week Monday
df_fact["week_start_date"] = (
    df_fact["fecha_albaran"] - pd.to_timedelta(df_fact["fecha_albaran"].dt.weekday, unit="D")
).dt.normalize()

# Aggregate
agg_dict = {
    "y_sales":          ("unidades",        lambda x: x.clip(lower=0).sum()),
    "base_week_total":  ("base_imponible",  "sum"),
}
if HAS_COSTE:
    agg_dict["coste_week_total"] = ("importe_coste", "sum")

df_weekly = (
    df_fact
    .groupby(["sku_id","week_start_date"])
    .agg(**agg_dict)
    .reset_index()
)

# Unit price (safe div)
units_pos = df_weekly["y_sales"].clip(lower=1)
df_weekly["unit_price_net_week"] = df_weekly["base_week_total"] / units_pos

share_pos = (df_weekly["y_sales"] > 0).mean()
print(f"Weekly rows    : {len(df_weekly):,}")
print(f"Unique SKUs    : {df_weekly['sku_id'].nunique():,}")
print(f"share(y_sales>0): {share_pos:.3%}")
print(df_weekly["y_sales"].describe())

assert share_pos > 0.01, (
    f"FATAL — share(y_sales>0)={share_pos:.4f}. "
    "Check that unidades was parsed correctly. "
    f"Sample: {df_weekly['y_sales'].head(20).tolist()}"
)

## Cell 4 — Data-Driven Season Group (feature, not filter)

In [ ]:
df_weekly["month"] = df_weekly["week_start_date"].dt.month

monthly_units = df_weekly.groupby("month")["y_sales"].sum().sort_values(ascending=False)
N_HIGH = 5
SEASON_MONTHS = set(monthly_units.head(N_HIGH).index.tolist())

df_weekly["season_group"] = df_weekly["month"].apply(
    lambda m: "HIGH_SEASON" if m in SEASON_MONTHS else "REST"
)

print(f"HIGH_SEASON months : {sorted(SEASON_MONTHS)}")
print("Monthly distribution:")
print(df_weekly.groupby("month")[["y_sales","season_group"]].agg(
    total_units=("y_sales","sum"), season=("season_group","first")
).to_string())

## Cell 5 — Feature Engineering (lags, rolling, amplitude)

In [ ]:
df_weekly = df_weekly.sort_values(["sku_id","week_start_date"]).copy()

grp = df_weekly.groupby("sku_id")["y_sales"]

df_weekly["lag_1"]  = grp.shift(1)
df_weekly["lag_2"]  = grp.shift(2)
df_weekly["lag_4"]  = grp.shift(4)
df_weekly["lag_13"] = grp.shift(13)

# Rolling (exclusive of current row: min_periods=1 on shifted series)
df_weekly["roll4_mean"]  = grp.shift(1).groupby(df_weekly["sku_id"]).transform(
    lambda x: x.rolling(4,  min_periods=1).mean()
)
df_weekly["roll13_mean"] = grp.shift(1).groupby(df_weekly["sku_id"]).transform(
    lambda x: x.rolling(13, min_periods=1).mean()
)
df_weekly["roll13_std"]  = grp.shift(1).groupby(df_weekly["sku_id"]).transform(
    lambda x: x.rolling(13, min_periods=2).std().fillna(0)
)

df_weekly["amplitude"]       = df_weekly["roll13_mean"]
df_weekly["cv_13"]           = (df_weekly["roll13_std"] / df_weekly["roll13_mean"].clip(lower=1e-6)).clip(upper=10)

# ever_sold_before = cumulative-max of y_sales>0 BEFORE this row
df_weekly["ever_sold_before"] = (
    df_weekly.groupby("sku_id")["y_sales"]
    .transform(lambda x: (x > 0).shift(1).fillna(0).cummax())
    .astype(int)
)

# Diagnostic
val_mask = (df_weekly["week_start_date"] >= VAL_START) & (df_weekly["week_start_date"] <= VAL_END)
for feat in ["lag_1","roll4_mean","roll13_mean","amplitude"]:
    all_nan = df_weekly.loc[val_mask, feat].isna().mean()
    all_zero = (df_weekly.loc[val_mask, feat] == 0).mean()
    print(f"  VAL {feat}: NaN={all_nan:.2%}  zero={all_zero:.2%}")

print(f"\nFeature engineering done. Columns: {len(df_weekly.columns)}")

## Cell 6 — Leakage-Safe Label Join at t+4 weeks

In [ ]:
# label_week = decision week + 4 weeks
df_weekly["label_week"] = df_weekly["week_start_date"] + pd.Timedelta(weeks=HORIZON_WEEKS)

# Self-join: for each (sku, label_week) find the actual sales at that week
sales_lookup = df_weekly[["sku_id","week_start_date","y_sales"]].rename(
    columns={"week_start_date": "label_week", "y_sales": "y_true_h4"}
)

df_model = df_weekly.merge(sales_lookup, on=["sku_id","label_week"], how="left")

before = len(df_model)
df_model = df_model[df_model["y_true_h4"].notna()].copy()
after  = len(df_model)
print(f"Rows with y_true_h4 found: {after:,} / {before:,}  (dropped {before-after:,})")

# Diagnostics by period
for period, mask in [
    ("TRAIN", df_model["label_week"] <  (VAL_START - pd.Timedelta(weeks=CALIB_WEEKS))),
    ("VAL",   (df_model["label_week"] >= VAL_START) & (df_model["label_week"] <= VAL_END)),
]:
    sub = df_model[mask]["y_true_h4"]
    if len(sub):
        print(f"  {period}: n={len(sub):,}  mean={sub.mean():.2f}  share>0={( sub>0).mean():.3%}")

val_share = (
    df_model[
        (df_model["label_week"] >= VAL_START) & (df_model["label_week"] <= VAL_END)
    ]["y_true_h4"] > 0
).mean()

assert val_share > 0, (
    "FATAL — y_true_h4 is 0 for 100% of VAL. "
    "Check that label_week (decision+4w) falls within the fact data date range. "
    f"fact date range: {df_fact['fecha_albaran'].min().date()} ~ {df_fact['fecha_albaran'].max().date()}"
)
print(f"\nLabel join OK. VAL share(y_true_h4>0) = {val_share:.3%}")

## Cell 7 — Time Splits with Guaranteed CALIB

In [ ]:
calib_start = VAL_START - pd.Timedelta(weeks=CALIB_WEEKS)
print(f"TRAIN  : label_week < {calib_start.date()}")
print(f"CALIB  : {calib_start.date()} <= label_week < {VAL_START.date()}")
print(f"VAL    : {VAL_START.date()} <= label_week <= {VAL_END.date()}")

df_model["split"] = "TRAIN"
df_model.loc[(df_model["label_week"] >= calib_start) &
             (df_model["label_week"] <  VAL_START),  "split"] = "CALIB"
df_model.loc[(df_model["label_week"] >= VAL_START) &
             (df_model["label_week"] <= VAL_END),    "split"] = "VAL"

split_counts = df_model["split"].value_counts()
print("\nSplit counts:")
print(split_counts.to_string())

assert split_counts.get("CALIB", 0) > 0, (
    f"FATAL — CALIB split has 0 rows. "
    f"calib_start={calib_start.date()}, VAL_START={VAL_START.date()}. "
    f"label_week range: {df_model['label_week'].min().date()} ~ {df_model['label_week'].max().date()}"
)
print("\nSplit assertion OK.")

## Cell 8 — OOS Proxy Label & Eval Labels

In [ ]:
# demand_recent: best single-number summary of expected demand before decision
df_model["demand_recent"] = df_model[["lag_1","roll4_mean","roll13_mean"]].max(axis=1).fillna(0)

# model label: stockout only for SKUs that were actively selling (avoid cold-start noise)
df_model["stockout_event_h4"] = (
    (df_model["y_true_h4"] == 0).astype(int)
    * df_model["ever_sold_before"]
    * (df_model["demand_recent"] > ACTIVE_THRESHOLD).astype(int)
)
df_model["true_stockout_label_model"] = df_model["stockout_event_h4"].astype(int)

# Control label (simpler): just sales==0
df_model["true_stockout_sales0"] = (df_model["y_true_h4"] == 0).astype(int)

print("Prevalence by split and label:")
for split in ["TRAIN","CALIB","VAL"]:
    mask = df_model["split"] == split
    n    = mask.sum()
    if n == 0:
        continue
    p_model = df_model.loc[mask, "true_stockout_label_model"].mean()
    p_ctrl  = df_model.loc[mask, "true_stockout_sales0"].mean()
    print(f"  {split:5s} n={n:7,}  model_label={p_model:.4f}  sales0={p_ctrl:.4f}")

val_prev = df_model.loc[df_model["split"] == "VAL", "true_stockout_label_model"].mean()
assert 0.001 < val_prev < 0.999, (
    f"FATAL — model label prevalence in VAL = {val_prev:.4f}. "
    "Degenerate label. Check ACTIVE_THRESHOLD or ever_sold_before. "
    "Try lowering ACTIVE_THRESHOLD."
)
print(f"\nLabel prevalence check OK (VAL={val_prev:.4f}).")

## Cell 9 — Layer 1: OOS Classifier → p_oos_h4

In [ ]:
FEATURES = [
    "lag_1","lag_2","lag_4","lag_13",
    "roll4_mean","roll13_mean","roll13_std",
    "amplitude","cv_13",
    "ever_sold_before","demand_recent",
    "month",
]
FEATURES = [f for f in FEATURES if f in df_model.columns]
print(f"Features ({len(FEATURES)}): {FEATURES}")

train_calib_mask = df_model["split"].isin(["TRAIN","CALIB"])
calib_mask       = df_model["split"] == "CALIB"
val_mask         = df_model["split"] == "VAL"

X_tc = df_model.loc[train_calib_mask, FEATURES].fillna(0)
y_tc = df_model.loc[train_calib_mask, "true_stockout_label_model"]
X_c  = df_model.loc[calib_mask,       FEATURES].fillna(0)
y_c  = df_model.loc[calib_mask,       "true_stockout_label_model"]

# HGBClassifier (handles NaN natively, but we filled above for safety)
clf = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=50, class_weight="balanced", random_state=42,
)
clf.fit(X_tc, y_tc)
print("Layer1 classifier trained.")

# Raw probabilities for all rows
X_all = df_model[FEATURES].fillna(0)
p_raw = clf.predict_proba(X_all)[:, 1]
df_model["p_oos_raw"] = p_raw

# Platt calibration on CALIB
if calib_mask.sum() > 50:
    platt = LogisticRegression(C=1.0, max_iter=500, random_state=42)
    platt.fit(p_raw[calib_mask.values].reshape(-1,1), y_c)
    p_cal = platt.predict_proba(p_raw.reshape(-1,1))[:,1]
    df_model["p_oos_h4"] = p_cal
    print("Platt calibration applied.")
else:
    df_model["p_oos_h4"] = p_raw
    print("Platt skipped (CALIB too small); using raw probabilities.")

# Diagnostics
val_nunique = df_model.loc[val_mask, "p_oos_h4"].nunique()
print(f"p_oos_h4 nunique in VAL = {val_nunique}")

try:
    auc = roc_auc_score(
        df_model.loc[val_mask, "true_stockout_label_model"],
        df_model.loc[val_mask, "p_oos_h4"]
    )
    ll  = log_loss(
        df_model.loc[val_mask, "true_stockout_label_model"],
        df_model.loc[val_mask, "p_oos_h4"]
    )
    print(f"VAL AUC={auc:.4f}  LogLoss={ll:.4f}")
except Exception as e:
    print(f"AUC/LogLoss skipped: {e}")

assert val_nunique > 10, (
    f"FATAL — p_oos_h4 is near-constant in VAL (nunique={val_nunique}). "
    "Check feature variance and label balance."
)
print("Layer1 assertion OK.")

## Cell 10 — Layer 2: Demand Regressor → yhat_p50_h4

In [ ]:
FEATURES_REG = FEATURES + ["p_oos_h4"]

X_tc_r = df_model.loc[train_calib_mask, FEATURES_REG].fillna(0)
y_tc_r = df_model.loc[train_calib_mask, "y_true_h4"]
X_all_r = df_model[FEATURES_REG].fillna(0)

reg = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
    min_samples_leaf=20, loss="squared_error", random_state=42,
)
reg.fit(X_tc_r, y_tc_r)
print("Layer2 regressor trained.")

df_model["yhat_p50_h4"] = np.clip(reg.predict(X_all_r), 0, None)

val_median = df_model.loc[val_mask, "yhat_p50_h4"].median()
print(f"yhat_p50_h4 median in VAL = {val_median:.4f}")
print(df_model.loc[val_mask, "yhat_p50_h4"].describe())

assert val_median > 0, (
    f"FATAL — yhat_p50_h4 median in VAL = {val_median:.4f}. "
    "All predictions are 0. Check y_true_h4 distribution in TRAIN."
)
print("Layer2 assertion OK.")

## Cell 11 — Layer 3: Conformal Quantiles → q90/q95
> Calibrated on CALIB active rows; bin edges derived from CALIB only.

In [ ]:
# Scale
df_model["scale"] = np.maximum(
    1.0,
    np.maximum(
        df_model["roll13_std"].fillna(0),
        np.sqrt((df_model["roll13_mean"].fillna(0) + 1.0).clip(lower=0))
    )
)

# Conformal scores on CALIB
calib_idx = df_model[calib_mask].index
df_model.loc[calib_idx, "conf_score"] = (
    (df_model.loc[calib_idx, "y_true_h4"] - df_model.loc[calib_idx, "yhat_p50_h4"])
    / df_model.loc[calib_idx, "scale"]
)
# NaN out inactive rows (no demand history)
df_model.loc[df_model["amplitude"] < ACTIVE_THRESHOLD, "conf_score"] = np.nan

# Demand decile thresholds from CALIB only
calib_active = df_model.loc[calib_mask & (df_model["amplitude"] >= ACTIVE_THRESHOLD)]
N_DECILES = 5
if len(calib_active) > N_DECILES * 10:
    _, bin_edges = pd.qcut(
        calib_active["amplitude"], q=N_DECILES, retbins=True, duplicates="drop"
    )
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
else:
    bin_edges = [-np.inf, np.inf]
    print(f"  Too few CALIB active rows ({len(calib_active):,}) — using single demand bin.")

df_model["demand_decile"] = pd.cut(
    df_model["amplitude"], bins=bin_edges, labels=False, include_lowest=True
).fillna(0).astype(int)

df_model["segment"] = df_model["season_group"] + "_D" + df_model["demand_decile"].astype(str)

# Quantiles per segment on CALIB
seg_quantiles = {}
for seg, grp in df_model[calib_mask].groupby("segment"):
    scores = grp["conf_score"].dropna().values
    if len(scores) >= 5:
        seg_quantiles[seg] = {
            "q90": float(np.nanquantile(scores, 0.90)),
            "q95": float(np.nanquantile(scores, 0.95)),
        }

print(f"Segments with quantiles: {len(seg_quantiles)}")

# Fallback global quantiles
global_scores = df_model.loc[calib_mask, "conf_score"].dropna().values
fallback = {
    "q90": float(np.nanquantile(global_scores, 0.90)) if len(global_scores) else 0.0,
    "q95": float(np.nanquantile(global_scores, 0.95)) if len(global_scores) else 0.0,
}
print(f"Fallback quantiles: q90={fallback['q90']:.3f}  q95={fallback['q95']:.3f}")

def get_q(row, level):
    return seg_quantiles.get(row["segment"], fallback)[level]

df_model["q_score_p90"] = df_model.apply(lambda r: get_q(r, "q90"), axis=1)
df_model["q_score_p95"] = df_model.apply(lambda r: get_q(r, "q95"), axis=1)

df_model["q90_h4"] = (df_model["yhat_p50_h4"] + df_model["q_score_p90"] * df_model["scale"]).clip(lower=0)
df_model["q95_h4"] = (df_model["yhat_p50_h4"] + df_model["q_score_p95"] * df_model["scale"])
df_model["q95_h4"] = df_model[["q90_h4","q95_h4"]].max(axis=1)  # monotone

# ── B3 Conditional coverage gate ─────────────────────────────────────────────
print("\nB3 Conditional Coverage (VAL active rows):")
b3_rows = []
for sg in df_model["season_group"].unique():
    mask_sg = val_mask & (df_model["season_group"] == sg) & (df_model["amplitude"] >= ACTIVE_THRESHOLD)
    sub = df_model[mask_sg]
    if len(sub) == 0:
        continue
    viol90 = (sub["y_true_h4"] > sub["q90_h4"]).mean()
    b3_rows.append({"season_group": sg, "n": len(sub), "viol_p90": round(viol90,4),
                    "B3": "PASS" if viol90 <= 0.12 else "FAIL"})
    print(f"  {sg}: n={len(sub):,}  viol_p90={viol90:.4f}  -> {'PASS' if viol90<=0.12 else 'FAIL'}")

df_b3 = pd.DataFrame(b3_rows)
b3_overall = (df_b3["B3"] == "PASS").all() if len(df_b3) else False
print(f"\nB3 Gate: {'PASS' if b3_overall else 'CONDITIONAL_PASS (check viol rates)'}")

## Cell 11b — B3 Tuning: Val Tune/Test Split & Correction Factors
> Fix B3 violations by scaling quantile width per season_group

In [ ]:
# ── Task A1: Active validation set ──────────────────────────────────────────
active_val_mask = val_mask & (df_model["amplitude"] >= ACTIVE_THRESHOLD)
print(f"Active VAL rows: {active_val_mask.sum():,}")

# ── Task A2: Quantile width above point forecast ────────────────────────────
df_model["delta90"] = df_model["q90_h4"] - df_model["yhat_p50_h4"]
df_model["delta95"] = df_model["q95_h4"] - df_model["yhat_p50_h4"]

pos_delta = (df_model.loc[active_val_mask, "delta90"] > 0).mean()
print(f"share(delta90>0) in active VAL: {pos_delta:.3%}")
assert pos_delta > 0.9, f"delta90 not positive for {1-pos_delta:.1%} of active VAL"

# ── Task A3: VAL tune/test split by season ──────────────────────────────────
df_model["is_tune"] = False
for sg in df_model["season_group"].unique():
    sg_weeks = (
        df_model[(df_model["split"] == "VAL") & (df_model["season_group"] == sg)]
        ["week_start_date"].drop_duplicates().sort_values()
    )
    if len(sg_weeks) < 3:
        # Too few weeks, use all as tune
        tune_weeks = set(sg_weeks)
    else:
        n_tune = max(1, int(len(sg_weeks) * 2 / 3))
        tune_weeks = set(sg_weeks.iloc[:n_tune])
    
    mask = (df_model["split"] == "VAL") & (df_model["season_group"] == sg) & (df_model["week_start_date"].isin(tune_weeks))
    df_model.loc[mask, "is_tune"] = True

tune_n = (df_model["split"] == "VAL") & df_model["is_tune"]
test_n = (df_model["split"] == "VAL") & (~df_model["is_tune"])
print(f"VAL split: TUNE={tune_n.sum():,}  TEST={test_n.sum():,}")

In [ ]:
# ── Task A4: Find correction factors by season ──────────────────────────────
def find_cf_binary_search(season_grp, df, target_viol=0.10, tol=0.005):
    """Binary search for correction factor to achieve target violation rate."""
    mask = (
        (df["split"] == "VAL") & 
        df["is_tune"] & 
        (df["season_group"] == season_grp) & 
        (df["amplitude"] >= ACTIVE_THRESHOLD)
    )
    sub = df[mask]
    if len(sub) < 50:
        return 1.0, len(sub), np.nan, np.nan
    
    y_true = sub["y_true_h4"].values
    yhat = sub["yhat_p50_h4"].values
    delta = sub["delta90"].values
    
    def viol_rate(cf):
        q90_adj = yhat + cf * delta
        return (y_true > q90_adj).mean()
    
    # Initial violation rate (cf=1.0)
    viol_before = viol_rate(1.0)
    
    # Binary search: viol decreases as cf increases
    # Goal: viol_rate <= 0.12, prefer close to 0.10
    cf_low, cf_high = 0.5, 5.0
    best_cf = 1.0
    
    for _ in range(20):
        cf_mid = (cf_low + cf_high) / 2
        viol = viol_rate(cf_mid)
        
        if abs(viol - target_viol) < tol:
            best_cf = cf_mid
            break
        
        if viol > target_viol:
            # Need wider interval
            cf_low = cf_mid
        else:
            # Too wide, can narrow
            cf_high = cf_mid
        best_cf = cf_mid
    
    # Ensure we don't exceed 0.12
    viol_final = viol_rate(best_cf)
    if viol_final > 0.12:
        # Widen more
        for cf_try in np.linspace(best_cf, 5.0, 10):
            if viol_rate(cf_try) <= 0.12:
                best_cf = cf_try
                break
    
    viol_after = viol_rate(best_cf)
    return best_cf, len(sub), viol_before, viol_after

tuning_factors = []
for sg in df_model["season_group"].unique():
    cf, n, viol_b, viol_a = find_cf_binary_search(sg, df_model)
    tuning_factors.append({
        "season_group": sg,
        "cf": round(cf, 3),
        "n_tune": n,
        "viol_before": round(viol_b, 4) if not np.isnan(viol_b) else np.nan,
        "viol_after": round(viol_a, 4) if not np.isnan(viol_a) else np.nan,
    })

df_cf_factors = pd.DataFrame(tuning_factors)
print("\nB3 Tuning Factors (by season):")
print(df_cf_factors.to_string(index=False))

In [ ]:
# ── Task A5: Apply correction factors to all VAL ────────────────────────────
cf_map = dict(zip(df_cf_factors["season_group"], df_cf_factors["cf"]))

df_model["cf_season"] = df_model["season_group"].map(cf_map).fillna(1.0)
df_model["q90_h4_v3"] = df_model["yhat_p50_h4"] + df_model["cf_season"] * df_model["delta90"]
df_model["q95_h4_v3"] = df_model["yhat_p50_h4"] + df_model["cf_season"] * df_model["delta95"]

# Enforce bounds
df_model["q90_h4_v3"] = df_model["q90_h4_v3"].clip(lower=0)
df_model["q95_h4_v3"] = df_model[["q90_h4_v3", "q95_h4_v3"]].max(axis=1)

# Re-evaluate B3 on full VAL active
print("\nB3 After Tuning (full VAL active):")
b3_v3_rows = []
for sg in df_model["season_group"].unique():
    mask_sg = val_mask & (df_model["season_group"] == sg) & (df_model["amplitude"] >= ACTIVE_THRESHOLD)
    sub = df_model[mask_sg]
    if len(sub) == 0:
        continue
    viol90_v3 = (sub["y_true_h4"] > sub["q90_h4_v3"]).mean()
    b3_v3_rows.append({
        "season_group": sg,
        "n": len(sub),
        "viol_p90_v3": round(viol90_v3, 4),
        "B3_v3": "PASS" if viol90_v3 <= 0.12 else "FAIL"
    })
    print(f"  {sg}: n={len(sub):,}  viol_p90={viol90_v3:.4f}  -> {'PASS' if viol90_v3<=0.12 else 'FAIL'}")

df_b3_v3 = pd.DataFrame(b3_v3_rows)
b3_v3_overall = (df_b3_v3["B3_v3"] == "PASS").all() if len(df_b3_v3) else False
print(f"\nB3 Gate v3: {'PASS' if b3_v3_overall else 'FAIL'}")

assert b3_v3_overall, "FATAL — B3 still failing after tuning. Check correction factors or data."

# Save tuning report
df_cf_factors.to_csv(REPORT_DIR / "b3_tuning_factors.csv", index=False)
df_b3_v3.to_csv(REPORT_DIR / "eval_coverage_h4_conditional_v3.csv", index=False)
print(f"\nSaved: b3_tuning_factors.csv, eval_coverage_h4_conditional_v3.csv")

## Cell 12 — Build Forecast DataFrame & Save

In [ ]:
FORECAST_COLS = [
    "sku_id", "week_start_date", "label_week", "season_group", "split", "is_tune",
    "y_true_h4", "y_sales",
    "p_oos_h4", "yhat_p50_h4", "q90_h4", "q95_h4", "q90_h4_v3", "q95_h4_v3",
    "amplitude", "scale", "demand_recent", "ever_sold_before", "cf_season",
    "true_stockout_label_model", "true_stockout_sales0",
]
FORECAST_COLS = [c for c in FORECAST_COLS if c in df_model.columns]

df_forecast = df_model[FORECAST_COLS].copy()
out_path = OUT_DIR / "forecast_h4_enriched.csv"
df_forecast.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({out_path.stat().st_size:,} bytes  {len(df_forecast):,} rows)")

## Cell 13 — Economic Enrichment (price / margin) — pure pandas, no BQ

In [ ]:
# Build KPI per SKU from fact
kpi_agg = df_fact.groupby("sku_id" if "sku_id" in df_fact.columns else "codigo_articulo").apply(
    lambda g: pd.Series({
        "precio_unit_net": g["base_imponible"].sum() / g["unidades"].clip(lower=0).sum()
            if g["unidades"].clip(lower=0).sum() > 0 else np.nan,
        "margen_unit": (
            (g["base_imponible"].sum() - g["importe_coste"].sum()) / g["unidades"].clip(lower=0).sum()
            if HAS_COSTE and g["unidades"].clip(lower=0).sum() > 0 else np.nan
        ),
    })
).reset_index()
kpi_agg.columns = ["sku_id","precio_unit_net","margen_unit"]

if "margen_unit" in kpi_agg.columns:
    kpi_agg["margen_unit"] = kpi_agg["margen_unit"].clip(lower=0)

print(f"KPI rows: {len(kpi_agg):,}")
print(kpi_agg[["precio_unit_net","margen_unit"]].describe())

# Merge into forecast
if "sku_id" not in df_forecast.columns and "codigo_articulo" in df_forecast.columns:
    kpi_agg = kpi_agg.rename(columns={"sku_id":"codigo_articulo"})
    df_forecast = df_forecast.merge(kpi_agg, on="codigo_articulo", how="left")
else:
    df_forecast = df_forecast.merge(kpi_agg, on="sku_id", how="left")

# Risk scores
p  = df_forecast["p_oos_h4"].fillna(0)
y  = df_forecast["yhat_p50_h4"].fillna(0)
pr = df_forecast.get("precio_unit_net", pd.Series(np.nan, index=df_forecast.index)).fillna(0)
mg = df_forecast.get("margen_unit",      pd.Series(np.nan, index=df_forecast.index)).fillna(0)
q9 = df_forecast["q90_h4"].fillna(0)
q95= df_forecast["q95_h4"].fillna(0)

df_forecast["riesgo_stockout_eur"] = p * y   * pr
df_forecast["riesgo_q95_eur"]      = p * q95 * pr
df_forecast["eur_at_risk_margin"]  = p * q9  * mg if HAS_COSTE else np.nan

out_path = OUT_DIR / "forecast_h4_enriched.csv"
df_forecast.to_csv(out_path, index=False)
print(f"\nEnriched forecast saved: {out_path}  ({len(df_forecast):,} rows)")
print(f"  riesgo_stockout_eur sum = {df_forecast['riesgo_stockout_eur'].sum():,.0f}")

## Cell 13b — B4 Policy Selection (objective: maximise € recuperable)
> Multi-objective: primary = `eur_expected_tp_top100` (p × q95v3 × price); constraint = `lift@100 >= LIFT_MIN` (baseline floor on VAL_TUNE); tie-breaks = `eur_severity_tp_top100`, `eur_precision_expected`.

In [ ]:
# ── B4 Cell 1: Flags, helpers, eval_policy_euro_extended ────────────────────
HAS_PRECIO = "precio_unit_net"  in df_forecast.columns
HAS_MARGEN = "margen_unit"      in df_forecast.columns
HAS_WIDTH  = "uncertainty_width" in df_forecast.columns

if not HAS_PRECIO:
    raise RuntimeError(
        "precio_unit_net not found in df_forecast. "
        "Add a price column before running B4."
    )

def topk_by_week(df, score_col, k=100, week_col="week_start_date"):
    """Return top-k rows per decision week sorted by score descending."""
    return (
        df.sort_values(score_col, ascending=False)
        .groupby(week_col, group_keys=False)
        .head(k)
        .reset_index(drop=True)
    )

def eval_policy_euro_extended(
    df_universe,
    score_col,
    label_col="true_stockout_label_model",
    k=100,
):
    """
    Evaluate a ranking policy by season_group + ALL.

    Columns required in df_universe: eur_expected, eur_severity.

    Returns per segment:
      n_alerts, n_tp, precision@100, recall@100, lift@100
      eur_expected_sum_top100, eur_expected_tp_top100, eur_precision_expected
      eur_severity_sum_top100, eur_severity_tp_top100, eur_precision_severity
    """
    top100 = topk_by_week(df_universe, score_col, k=k)
    rows = []
    for seg in list(df_universe["season_group"].unique()) + ["ALL"]:
        u = df_universe if seg == "ALL" else df_universe[df_universe["season_group"] == seg]
        a = top100       if seg == "ALL" else top100[top100["season_group"] == seg]

        n_u   = len(u)
        n_a   = len(a)
        n_pos = float(u[label_col].sum())
        n_tp  = float(a[label_col].sum())

        prec = n_tp / n_a   if n_a  > 0 else 0.0
        rec  = n_tp / n_pos if n_pos > 0 else 0.0
        base = n_pos / n_u  if n_u  > 0 else 0.0
        lift = prec  / base if base  > 0 else 0.0

        # € expected  (probability × q95v3 × price)
        e_sum  = float(a["eur_expected"].sum())
        e_tp   = float((a["eur_expected"] * a[label_col]).sum())
        e_prec = e_tp / e_sum if e_sum > 0 else 0.0

        # € severity  (q95v3 × price)
        s_sum  = float(a["eur_severity"].sum())
        s_tp   = float((a["eur_severity"] * a[label_col]).sum())
        s_prec = s_tp / s_sum if s_sum > 0 else 0.0

        rows.append({
            "season_group":             seg,
            "n_alerts":                 n_a,
            "n_tp":                     int(n_tp),
            "precision@100":            round(prec,   4),
            "recall@100":               round(rec,    4),
            "lift@100":                 round(lift,   2),
            "eur_expected_sum_top100":  round(e_sum,  2),
            "eur_expected_tp_top100":   round(e_tp,   2),
            "eur_precision_expected":   round(e_prec, 4),
            "eur_severity_sum_top100":  round(s_sum,  2),
            "eur_severity_tp_top100":   round(s_tp,   2),
            "eur_precision_severity":   round(s_prec, 4),
        })
    return pd.DataFrame(rows)

print(f"B4 helpers v2 ready.  HAS_PRECIO={HAS_PRECIO}  HAS_MARGEN={HAS_MARGEN}  HAS_WIDTH={HAS_WIDTH}")

In [ ]:
# ── B4 Cell 2: Build df_val + business columns + candidate score columns ─────
df_val = df_forecast[df_forecast["split"] == "VAL"].copy()

p    = df_val["p_oos_h4"].fillna(0)
y    = df_val["yhat_p50_h4"].fillna(0)
pr   = df_val["precio_unit_net"].fillna(0)
q90v = df_val["q90_h4_v3"].fillna(0)
q95v = df_val["q95_h4_v3"].fillna(0)

# Business objective columns (used inside eval_policy_euro_extended)
df_val["eur_expected"] = p * q95v * pr       # probability × q95v3 × price
df_val["eur_severity"] = q95v * pr           # severity only (no probability compression)

# ── Candidate ranking scores ──────────────────────────────────────────────────
df_val["score_A_baseline"] = p * y    * pr
df_val["score_C_q90v3"]    = p * q90v * pr
df_val["score_D_q95v3"]    = p * q95v * pr

if HAS_WIDTH:
    uw = df_val["uncertainty_width"].fillna(0)
else:
    uw = (q95v - y).clip(lower=0)            # proxy: interval half-width
    df_val["uncertainty_width"] = uw
df_val["score_G_width15"] = (p**1.5) * uw * pr

if HAS_MARGEN:
    mg = df_val["margen_unit"].fillna(0)
    df_val["score_M_margin"] = p * q90v * mg

score_cols = [c for c in df_val.columns if c.startswith("score_")]
print(f"eur_expected median : {df_val['eur_expected'].median():.3f}")
print(f"eur_severity median : {df_val['eur_severity'].median():.3f}")
print(f"Candidate scores ({len(score_cols)}): {score_cols}")

In [ ]:
# ── B4 Cell 3: Hybrid score grid + full sweep + multi-objective selection ─────

# ── 1. Width column (may already exist from Cell 2; reuse or recompute) ───────
if "uncertainty_width" in df_val.columns:
    _width_val = df_val["uncertainty_width"].fillna(0)
else:
    _width_val = (df_val["q95_h4_v3"].fillna(0) - df_val["yhat_p50_h4"].fillna(0)).clip(lower=0)
    df_val["uncertainty_width"] = _width_val

# ── 2. Add hybrid score columns to df_val ─────────────────────────────────────
# score_H(γ, λ) = (p**γ) * y * pr  +  λ * p * width * pr
GAMMA_GRID  = [0.5, 1.0, 1.5, 2.0]
LAMBDA_GRID = [0.0, 0.25, 0.5, 1.0, 2.0]

_p  = df_val["p_oos_h4"].fillna(0)
_y  = df_val["yhat_p50_h4"].fillna(0)
_pr = df_val["precio_unit_net"].fillna(0)
_w  = df_val["uncertainty_width"].fillna(0)

hybrid_score_cols = []
for _g in GAMMA_GRID:
    for _l in LAMBDA_GRID:
        _col = f"score_H_g{_g}_l{_l}"
        df_val[_col] = (_p ** _g) * _y * _pr + _l * _p * _w * _pr
        hybrid_score_cols.append(_col)

all_score_cols = score_cols + hybrid_score_cols
print(f"Base scores   : {len(score_cols)}")
print(f"Hybrid scores : {len(hybrid_score_cols)}  (γ={GAMMA_GRID}, λ={LAMBDA_GRID})")
print(f"Total sweep   : {len(all_score_cols)} policies")

# ── 3. Rebuild df_tune / df_test from df_val (now contains all score columns) ─
df_tune = df_val[df_val["is_tune"] == True].copy()
df_test = df_val[df_val["is_tune"] == False].copy()

for _sg in df_val["season_group"].unique():
    assert (df_tune["season_group"] == _sg).sum() > 0, \
        f"FATAL — VAL_TUNE has 0 rows for season_group={_sg}"
print(f"\nVAL_TUNE: {len(df_tune):,}  |  VAL_TEST: {len(df_test):,}")

# ── 4. Sweep on VAL_TUNE ──────────────────────────────────────────────────────
_sweep_rows = []
for _sc in all_score_cols:
    _r = eval_policy_euro_extended(df_tune, _sc)
    _r.insert(0, "score_name", _sc)
    _sweep_rows.append(_r)
policy_sweep_hybrid = pd.concat(_sweep_rows, ignore_index=True)
policy_sweep_hybrid.to_csv(REPORT_DIR / "policy_sweep_hybrid_results.csv", index=False)
print("Saved: policy_sweep_hybrid_results.csv")

# ── 5. Multi-objective selection (ALL, VAL_TUNE) ──────────────────────────────
# STEP 1: Extract ALL rows
if "subset" in policy_sweep_hybrid.columns:
    _tune_rows = policy_sweep_hybrid[policy_sweep_hybrid["subset"].str.upper() == "VAL_TUNE"]
else:
    _tune_rows = policy_sweep_hybrid
df_sweep_all = _tune_rows[_tune_rows["season_group"] == "ALL"].copy()

# STEP 2: LIFT_MIN from score_A_baseline
_base_mask = df_sweep_all["score_name"] == "score_A_baseline"
if not _base_mask.any():
    raise RuntimeError("score_A_baseline not found in sweep table.")
LIFT_MIN = float(df_sweep_all.loc[_base_mask, "lift@100"].values[0])
print(f"\nLIFT_MIN (score_A_baseline, VAL_TUNE, ALL) = {LIFT_MIN:.2f}x")

# STEP 3: Constraint → primary → tie-breaks
# PRIMARY  : maximize eur_expected_sum_top100
# TIE-1    : maximize eur_severity_tp_top100
# TIE-2    : maximize eur_precision_expected
candidates = df_sweep_all[df_sweep_all["lift@100"] >= LIFT_MIN].copy()
_constrained = len(candidates) > 0
if not _constrained:
    print("WARNING ⚠️  — no policy meets LIFT_MIN; falling back to unconstrained.")
    candidates = df_sweep_all.copy()

candidates = candidates.sort_values(
    ["eur_expected_sum_top100", "eur_severity_tp_top100", "eur_precision_expected"],
    ascending=False,
).reset_index(drop=True)

best_score = candidates.iloc[0]["score_name"]
_best = candidates.iloc[0]

_disp_cols = [c for c in [
    "score_name", "eur_expected_sum_top100", "eur_expected_tp_top100",
    "eur_severity_tp_top100", "eur_precision_expected", "lift@100", "precision@100",
] if c in candidates.columns]
print("\nTop-5 candidates (sorted by primary objective):")
print(candidates[_disp_cols].head(5).to_string(index=False))

print(f"\n★ Selected policy : {best_score}  ({'constrained' if _constrained else 'UNCONSTRAINED fallback'})")
print(f"  eur_expected_sum = {_best['eur_expected_sum_top100']:,.2f}")
print(f"  eur_severity_tp  = {_best['eur_severity_tp_top100']:,.2f}")
print(f"  lift@100         = {_best['lift@100']:.2f}x")

In [ ]:
# ── B4 Cell 4: Validate chosen policy on VAL_TEST and full VAL ───────────────
eval_best_test = eval_policy_euro_extended(df_test, best_score)
eval_best_full = eval_policy_euro_extended(df_val,  best_score)

print(f"Chosen policy = {best_score}")
print("\nVAL_TEST:")
print(eval_best_test.to_string(index=False))
print("\nFull VAL:")
print(eval_best_full.to_string(index=False))

# Quality assertion: must have non-zero € exposure on full VAL
all_row_full = eval_best_full[eval_best_full["season_group"] == "ALL"].iloc[0]
eur_sum_chk  = all_row_full["eur_expected_sum_top100"]
assert eur_sum_chk > 0, f"FATAL — eur_expected_sum_top100 = {eur_sum_chk:.2f} on full VAL"
print(f"\n✅ eur_expected_sum_top100 = {eur_sum_chk:,.2f}")

# ── Comparison table: baseline vs chosen (VAL_TEST) ──────────────────────────
eval_baseline_test = eval_policy_euro_extended(df_test, "score_A_baseline")
cmp_rows = []
for sc, ev in [("score_A_baseline", eval_baseline_test), (best_score, eval_best_test)]:
    row = ev[ev["season_group"] == "ALL"].iloc[0].to_dict()
    row["score_name"] = sc
    cmp_rows.append(row)
df_cmp = pd.DataFrame(cmp_rows)[[
    "score_name", "lift@100",
    "eur_expected_tp_top100", "eur_severity_tp_top100",
    "eur_precision_expected", "precision@100",
]]
print("\nBaseline vs Chosen (VAL_TEST):")
print(df_cmp.to_string(index=False))

# Save per-segment evaluation for the chosen policy on full VAL
eval_out = eval_best_full.copy()
eval_out.insert(0, "score_name", best_score)
eval_out.to_csv(REPORT_DIR / "eval_alerts_top100_h4_pooled_bestpolicy.csv", index=False)
print("\nSaved: eval_alerts_top100_h4_pooled_bestpolicy.csv")

In [ ]:
# ── B4 Cell 5: Build alerts_top100_h4_bestpolicy.csv ─────────────────────────
alerts_best = topk_by_week(df_val, best_score, k=100).copy()
alerts_best["risk_score_eur"] = alerts_best[best_score]
alerts_best["eur_at_risk"]    = alerts_best[best_score]

keep_cols = [
    "week_start_date", "label_week", "sku_id", "season_group",
    "true_stockout_label_model",
    "p_oos_h4", "yhat_p50_h4", "q90_h4_v3", "q95_h4_v3",
    "precio_unit_net",
    "eur_expected", "eur_severity",
    "risk_score_eur", "eur_at_risk",
]
if HAS_MARGEN and "margen_unit" in alerts_best.columns:
    keep_cols.append("margen_unit")
if "is_tune" in alerts_best.columns:
    keep_cols.append("is_tune")

keep_cols = [c for c in keep_cols if c in alerts_best.columns]
alerts_best = alerts_best[keep_cols]

out_path = OUT_DIR / "alerts_top100_h4_bestpolicy.csv"
alerts_best.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(alerts_best):,} rows, {out_path.stat().st_size:,} bytes)")
print(alerts_best.head(3).to_string())

In [ ]:
# ── B4 Cell 6: Realistic-€ estimates + update run_summary_h4.csv ─────────────
summary_path = REPORT_DIR / "run_summary_h4.csv"
if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
else:
    df_summary = pd.DataFrame()

all_row_val = eval_best_full[eval_best_full["season_group"] == "ALL"].iloc[0]

# ── STEP 5: Realistic recoverable € estimates ─────────────────────────────────
# eur_expected_sum_top100 : expected € at risk in Top-100 (p × q95v3 × price)
# eur_severity_tp_top100  : upper bound if execution is perfect on all true positives
# eur_recuperable_realista: fraction of severity_tp realistically recovered
_exp_sum = float(all_row_val["eur_expected_sum_top100"])
_sev_tp  = float(all_row_val["eur_severity_tp_top100"])
_rec_50  = 0.5 * _sev_tp
_rec_70  = 0.7 * _sev_tp
_rec_90  = 0.9 * _sev_tp

print("=" * 70)
print("B4 — REALISMO ECONÓMICO (full VAL, ALL, best policy)")
print("=" * 70)
print(f"  eur_expected_sum_top100        : {_exp_sum:>12,.2f}  € expected at-risk in Top-100")
print(f"  eur_severity_tp_top100         : {_sev_tp:>12,.2f}  € upper bound (perfect execution on TPs)")
print(f"  eur_recuperable E=0.50 (realista baja) : {_rec_50:>12,.2f}")
print(f"  eur_recuperable E=0.70 (realista media): {_rec_70:>12,.2f}")
print(f"  eur_recuperable E=0.90 (realista alta) : {_rec_90:>12,.2f}")
print("=" * 70)

new_cols = {
    "chosen_policy_name":                   best_score,
    "LIFT_MIN":                             round(LIFT_MIN, 4),
    "eur_expected_sum_top100_ALL_VAL":      round(_exp_sum,  2),
    "eur_expected_tp_top100_ALL_VAL":       round(float(all_row_val.get("eur_expected_tp_top100", float("nan"))), 2),
    "eur_severity_tp_top100_ALL_VAL":       round(_sev_tp,   2),
    "eur_precision_expected_ALL_VAL":       round(float(all_row_val["eur_precision_expected"]),  4),
    "eur_recuperable_realista_E50":         round(_rec_50,   2),
    "eur_recuperable_realista_E70":         round(_rec_70,   2),
    "eur_recuperable_realista_E90":         round(_rec_90,   2),
    "b4_precision_at100":                   round(float(all_row_val["precision@100"]),  4),
    "b4_recall_at100":                      round(float(all_row_val["recall@100"]),     4),
    "b4_lift_at100":                        round(float(all_row_val["lift@100"]),       2),
}

if len(df_summary) > 0:
    for k, v in new_cols.items():
        df_summary[k] = v
else:
    df_summary = pd.DataFrame([new_cols])

df_summary.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")
print(pd.Series(new_cols).to_string())

## Impacto anual — inferencia de recuperación de ventas
> Proyección lineal de los resultados semanales Top-100 sobre un año completo (52 semanas). Extrapolación `annual_factor = 52 / VAL_weeks`.

In [ ]:
# ── Impacto anual — Parte 1: weekly timeseries ────────────────────────────────
_LABEL_COL = "true_stockout_label_model"

# Sanity: label must exist and have positive prevalence
if df_val[_LABEL_COL].isnull().all():
    raise RuntimeError("FATAL — true_stockout_label_model is all null in df_val.")
_prevalence = df_val[_LABEL_COL].mean()
if _prevalence == 0:
    raise RuntimeError(
        "FATAL — label prevalence is 0 in df_val. "
        "Cannot compute annual impact without positive labels."
    )
print(f"Label prevalence in VAL: {_prevalence:.3%}")

# Top-100 alerts per week on full VAL with the chosen policy
_alerts_annual = topk_by_week(df_val, best_score, k=100).copy()

# Per-week aggregation (all seasons + split by season_group)
_weekly_records = []
for _week, _grp in _alerts_annual.groupby("week_start_date"):
    _n_alerts = len(_grp)
    _n_tp     = int(_grp[_LABEL_COL].sum())
    _exp_sum  = float(_grp["eur_expected"].sum())
    _sev_tp   = float((_grp["eur_severity"] * _grp[_LABEL_COL]).sum())
    # dominant season in this week
    _season   = _grp["season_group"].mode().iloc[0]
    _weekly_records.append({
        "week_start_date":       _week,
        "season_group":          _season,
        "n_alerts":              _n_alerts,
        "n_tp":                  _n_tp,
        "eur_expected_sum_week": _exp_sum,
        "eur_severity_tp_week":  _sev_tp,
    })

df_weekly = (pd.DataFrame(_weekly_records)
             .sort_values("week_start_date")
             .reset_index(drop=True))

# Warnings
VAL_WEEKS = df_val["week_start_date"].nunique()
_zero_exp  = (df_weekly["eur_expected_sum_week"] <= 0).sum()
if _zero_exp > len(df_weekly) * 0.5:
    print(f"WARNING ⚠️  — {_zero_exp}/{len(df_weekly)} weeks have eur_expected_sum_week == 0")
if VAL_WEEKS < 30:
    print(f"WARNING ⚠️  — VAL covers only {VAL_WEEKS} weeks (<30). Annual extrapolation has high uncertainty.")
elif VAL_WEEKS > 53:
    print(f"WARNING ⚠️  — VAL covers {VAL_WEEKS} weeks (>53). Check for overlapping calendar years.")

df_weekly.to_csv(REPORT_DIR / "weekly_recovery_timeseries.csv", index=False)
print(f"\nSaved: weekly_recovery_timeseries.csv  ({len(df_weekly)} weeks)")
print(df_weekly[[
    "week_start_date", "season_group", "n_alerts", "n_tp",
    "eur_expected_sum_week", "eur_severity_tp_week",
]].head(6).to_string(index=False))

In [ ]:
# ── Impacto anual — Parte 2: annualization + run_summary + annual_impact_summary.md
import math as _math

ANNUAL_FACTOR = 52 / VAL_WEEKS
print(f"VAL_WEEKS = {VAL_WEEKS}  |  annual_factor = 52/{VAL_WEEKS} = {ANNUAL_FACTOR:.4f}")
if VAL_WEEKS != 52:
    print(f"  (Extrapolación lineal: resultados VAL × {ANNUAL_FACTOR:.4f})")

# ── Global (ALL) annualization ─────────────────────────────────────────────────
_exp_sum_val      = float(df_weekly["eur_expected_sum_week"].sum())
_sev_tp_val       = float(df_weekly["eur_severity_tp_week"].sum())

eur_expected_year     = _exp_sum_val  * ANNUAL_FACTOR
eur_severity_tp_year  = _sev_tp_val  * ANNUAL_FACTOR
eur_rec_E50_year      = 0.5 * eur_severity_tp_year
eur_rec_E70_year      = 0.7 * eur_severity_tp_year
eur_rec_E90_year      = 0.9 * eur_severity_tp_year

print("\n" + "=" * 70)
print("IMPACTO ANUAL — inferencia de recuperación de ventas (ALL seasons)")
print("=" * 70)
print(f"  eur_expected en VAL         : {_exp_sum_val:>14,.2f}")
print(f"  eur_severity_tp en VAL      : {_sev_tp_val:>14,.2f}")
print(f"  ── extrapolación × {ANNUAL_FACTOR:.4f} ─────────────────────────────────")
print(f"  eur_expected_year           : {eur_expected_year:>14,.2f}  €/año")
print(f"  eur_severity_tp_year        : {eur_severity_tp_year:>14,.2f}  €/año (upper bound)")
print(f"  eur_recuperable E=0.50/año  : {eur_rec_E50_year:>14,.2f}  €/año (realista baja)")
print(f"  eur_recuperable E=0.70/año  : {eur_rec_E70_year:>14,.2f}  €/año (realista media)")
print(f"  eur_recuperable E=0.90/año  : {eur_rec_E90_year:>14,.2f}  €/año (realista alta)")
print("=" * 70)

# ── By season_group ────────────────────────────────────────────────────────────
_by_season = (df_weekly
    .groupby("season_group", as_index=False)
    .agg(
        eur_expected_sum  =("eur_expected_sum_week", "sum"),
        eur_severity_tp   =("eur_severity_tp_week",  "sum"),
        n_weeks           =("week_start_date",        "count"),
    ))
_by_season["eur_expected_year"]    = _by_season["eur_expected_sum"] * ANNUAL_FACTOR
_by_season["eur_severity_tp_year"] = _by_season["eur_severity_tp"]  * ANNUAL_FACTOR
_by_season["eur_rec_E70_year"]     = 0.7 * _by_season["eur_severity_tp_year"]
print("\nPor season_group:")
print(_by_season[[
    "season_group", "n_weeks",
    "eur_expected_year", "eur_severity_tp_year", "eur_rec_E70_year",
]].to_string(index=False))

# ── Update run_summary_h4.csv ──────────────────────────────────────────────────
_summary_path = REPORT_DIR / "run_summary_h4.csv"
if _summary_path.exists():
    df_summary_ann = pd.read_csv(_summary_path)
else:
    df_summary_ann = pd.DataFrame()

_all_val_row = eval_best_full[eval_best_full["season_group"] == "ALL"].iloc[0]
_annual_cols = {
    "annual_factor":             round(ANNUAL_FACTOR, 6),
    "VAL_weeks":                 int(VAL_WEEKS),
    "eur_expected_year":         round(eur_expected_year,    2),
    "eur_severity_tp_year":      round(eur_severity_tp_year, 2),
    "eur_recuperable_E50_year":  round(eur_rec_E50_year,     2),
    "eur_recuperable_E70_year":  round(eur_rec_E70_year,     2),
    "eur_recuperable_E90_year":  round(eur_rec_E90_year,     2),
    "lift@100_ALL_VAL":          round(float(_all_val_row["lift@100"]),      2),
    "precision@100_ALL_VAL":     round(float(_all_val_row["precision@100"]), 4),
}
if len(df_summary_ann) > 0:
    for _k, _v in _annual_cols.items():
        df_summary_ann[_k] = _v
else:
    df_summary_ann = pd.DataFrame([_annual_cols])
df_summary_ann.to_csv(_summary_path, index=False)
print(f"\nSaved (updated): {_summary_path}")

# ── Write annual_impact_summary.md ─────────────────────────────────────────────
_b3_status = "✅ PASS" if b3_v3_overall else "❌ FAIL"
_sel_mode  = "Constrained (lift ≥ LIFT_MIN)" if _constrained else "⚠️ UNCONSTRAINED fallback"

_md_lines = [
    "# Impacto Anual — Recuperación de Ventas Cruzber (h=4)",
    "",
    f"**Policy seleccionado:** `{best_score}`  ",
    f"**Modo selección:** {_sel_mode}  ",
    f"**B3 Coverage Gate:** {_b3_status}  ",
    f"**Semanas en VAL:** {VAL_WEEKS}  |  **Factor anualización:** `52/{VAL_WEEKS} = {ANNUAL_FACTOR:.4f}`",
    "",
    "> **Advertencia:** Resultados basados en label proxy OOS (detección de stockouts vía ventas=0).",
    "> Extrapolación lineal `52/VAL_weeks`. Los valores anuales son estimaciones de impacto potencial,",
    "> no garantías de recuperación contable.",
    "",
    "## Tabla de impacto anual (ALL seasons)",
    "",
    "| Métrica | VAL total | Anualizado |",
    "|---------|-----------|------------|",
    f"| € esperado en Top-100 (ex-ante) | {_exp_sum_val:,.2f} | **{eur_expected_year:,.2f}** |",
    f"| € severidad TP (upper bound) | {_sev_tp_val:,.2f} | **{eur_severity_tp_year:,.2f}** |",
    f"| € recuperable E=0.50 (realista baja) | — | {eur_rec_E50_year:,.2f} |",
    f"| € recuperable E=0.70 (realista media) | — | **{eur_rec_E70_year:,.2f}** |",
    f"| € recuperable E=0.90 (realista alta) | — | {eur_rec_E90_year:,.2f} |",
    "",
    "## Por season_group",
    "",
    "| Season | Semanas | € expected/año | € severity_tp/año | € rec E=0.70/año |",
    "|--------|---------|----------------|-------------------|-----------------|",
]
for _, _srow in _by_season.iterrows():
    _md_lines.append(
        f"| {_srow['season_group']} | {int(_srow['n_weeks'])} "
        f"| {_srow['eur_expected_year']:,.2f} "
        f"| {_srow['eur_severity_tp_year']:,.2f} "
        f"| {_srow['eur_rec_E70_year']:,.2f} |"
    )

_md_lines += [
    "",
    "## Definiciones clave",
    "",
    "| Símbolo | Fórmula | Descripción |",
    "|---------|---------|-------------|",
    "| `eur_expected` | `p_oos_h4 × q95_h4_v3 × precio_unit_net` | € esperado ex-ante por SKU-semana |",
    "| `eur_severity_tp` | `q95_h4_v3 × precio_unit_net × label` | € real expuesto en TPs del Top-100 |",
    "| `E` | eficiencia operativa | Fracción de alertas TP convertidas en reposición exitosa |",
    "",
    f"*Generado automáticamente por WORKBENCH_H4_END2END.ipynb*",
]

_md_path = REPORT_DIR / "annual_impact_summary.md"
_md_path.write_text("\n".join(_md_lines), encoding="utf-8")
print(f"Saved: {_md_path}")

## Cell 14 — Alerts Top-100 per Decision Week

In [ ]:
df_forecast["risk_score"] = df_forecast["riesgo_stockout_eur"].fillna(
    df_forecast["p_oos_h4"]
)

alerts_top100 = (
    df_forecast
    .sort_values("risk_score", ascending=False)
    .groupby("week_start_date", group_keys=False)
    .head(100)
    .reset_index(drop=True)
)

out_path = OUT_DIR / "alerts_top100_h4.csv"
alerts_top100.to_csv(out_path, index=False)
print(f"Alerts saved : {out_path}  ({out_path.stat().st_size:,} bytes  {len(alerts_top100):,} rows)")
print(f"Weeks covered: {alerts_top100['week_start_date'].nunique()}")
print(alerts_top100[["sku_id","week_start_date","p_oos_h4","yhat_p50_h4","riesgo_stockout_eur"]].head(5).to_string())

## Cell 15 — Evaluation: Precision / Recall / Lift@100 (pooled by season)

In [ ]:
def evaluate_alerts(
    alerts_df: pd.DataFrame,
    universe_df: pd.DataFrame,
    label_col: str = "true_stockout_label_model",
) -> pd.DataFrame:
    """Compute pooled precision@k, recall@k, lift@k per season_group + ALL."""
    # Strict int conversion
    for df_ in [alerts_df, universe_df]:
        if label_col in df_.columns:
            df_[label_col] = (
                df_[label_col]
                .map(lambda x: 1 if str(x).lower() in ("1","true","yes") else 0)
                .astype(int)
            )

    # Null check
    null_rate = alerts_df[label_col].isna().mean() if label_col in alerts_df.columns else 1.0
    if null_rate > 0.5:
        print(f"  WARNING: {null_rate:.1%} null labels — evaluation may be unreliable")

    rows = []
    for seg in list(universe_df["season_group"].unique()) + ["ALL"]:
        u = universe_df if seg == "ALL" else universe_df[universe_df["season_group"] == seg]
        a = alerts_df   if seg == "ALL" else alerts_df[alerts_df["season_group"]   == seg]

        n_u   = len(u)
        n_a   = len(a)
        n_pos = u[label_col].sum()
        a_pos = a[label_col].sum()

        prec  = a_pos / n_a   if n_a  > 0 else np.nan
        rec   = a_pos / n_pos if n_pos > 0 else np.nan
        base  = n_pos / n_u   if n_u  > 0 else np.nan
        lift  = prec  / base  if (base is not np.nan and base > 0) else np.nan

        eur_col = "riesgo_stockout_eur"
        eur_sum = a[eur_col].sum() if eur_col in a.columns else np.nan

        rows.append({
            "season_group":         seg,
            "n_universe":           n_u,
            "n_alerts":             n_a,
            "n_positives_univ":     int(n_pos),
            "n_positives_alerts":   int(a_pos),
            "precision@100":        round(prec,  4) if not np.isnan(prec)  else np.nan,
            "recall@100":           round(rec,   4) if not np.isnan(rec)   else np.nan,
            "base_rate":            round(base,  4) if not np.isnan(base)  else np.nan,
            "lift@100":             round(lift,  2) if not np.isnan(lift)  else np.nan,
            "eur_at_risk_sum":      round(eur_sum, 0) if not np.isnan(eur_sum) else np.nan,
        })
    return pd.DataFrame(rows)


# VAL universe
val_universe = df_forecast[df_forecast["split"] == "VAL"].copy()
val_alerts   = alerts_top100[alerts_top100["week_start_date"].isin(
    val_universe["week_start_date"].unique()
)].copy()

df_eval = evaluate_alerts(val_alerts, val_universe)
print("Evaluation pooled:")
print(df_eval.to_string(index=False))

# ── Save reports ──────────────────────────────────────────────────────────────
df_eval.to_csv(REPORT_DIR / "eval_alerts_top100_h4_pooled.csv", index=False)
df_b3.to_csv(REPORT_DIR   / "eval_coverage_h4_conditional.csv", index=False)

# ── Run summary ───────────────────────────────────────────────────────────────
try:
    auc_val = roc_auc_score(
        val_universe["true_stockout_label_model"],
        val_universe["p_oos_h4"]
    )
except Exception:
    auc_val = np.nan

summary = pd.DataFrame([{
    "run_ts":            datetime.datetime.utcnow().isoformat(),
    "val_rows":          len(val_universe),
    "p_oos_nunique":     val_universe["p_oos_h4"].nunique(),
    "yhat_median":       val_universe["yhat_p50_h4"].median(),
    "q90_median":        val_universe["q90_h4"].median(),
    "share_q90_zero":    (val_universe["q90_h4"] == 0).mean(),
    "val_auc":           round(auc_val, 4) if not np.isnan(auc_val) else "n/a",
    "b3_pass":           b3_overall,
    "lift_ALL":          df_eval[df_eval["season_group"] == "ALL"]["lift@100"].values[0]
                         if "ALL" in df_eval["season_group"].values else np.nan,
    "eur_at_risk_total": df_eval[df_eval["season_group"] == "ALL"]["eur_at_risk_sum"].values[0]
                         if "ALL" in df_eval["season_group"].values else np.nan,
    "horizon_weeks":     HORIZON_WEEKS,
    "calib_weeks":       CALIB_WEEKS,
    "active_threshold":  ACTIVE_THRESHOLD,
}])
summary.to_csv(REPORT_DIR / "run_summary_h4.csv", index=False)
print(f"\nReports saved to {REPORT_DIR}/")

## Cell 16 — Final Sanity Summary

In [ ]:
val = df_forecast[df_forecast["split"] == "VAL"]

print("=" * 80)
print("  CRUZBER h=4 END-TO-END — RUN SUMMARY")
print("=" * 80)
print(f"  VAL rows              : {len(val):,}")
print(f"  VAL weeks             : {val['week_start_date'].nunique()}")
print(f"  p_oos_h4  nunique     : {val['p_oos_h4'].nunique()}")
print(f"  p_oos_h4  mean        : {val['p_oos_h4'].mean():.4f}")
print(f"  yhat_p50  median      : {val['yhat_p50_h4'].median():.2f}")
print(f"  q90_h4    median      : {val['q90_h4'].median():.2f}")
print(f"  q90_h4_v3 median      : {val['q90_h4_v3'].median():.2f}")
print(f"  share(q90_h4 == 0)    : {(val['q90_h4']==0).mean():.3%}")
print()

print("=" * 80)
print("B3 CONDITIONAL COVERAGE — BEFORE TUNING (baseline conformal)")
print("=" * 80)
print(df_b3.to_string(index=False))

print()
print("=" * 80)
print("B3 CONDITIONAL COVERAGE — AFTER TUNING (post-hoc correction factors)")
print("=" * 80)
print(df_b3_v3.to_string(index=False))
print(f"\n  ✅ B3 Gate Status: {'PASS' if b3_v3_overall else 'FAIL'}")

print()
print("=" * 80)
print("B4 HYBRID SWEEP — ALL, VAL_TUNE (top-5 by eur_expected_sum_top100)")
print("=" * 80)
_sweep_sorted = df_sweep_all.sort_values("eur_expected_sum_top100", ascending=False)
_sweep_disp_cols = [c for c in [
    "score_name", "eur_expected_sum_top100", "eur_severity_tp_top100",
    "eur_precision_expected", "lift@100", "precision@100",
] if c in _sweep_sorted.columns]
print(_sweep_sorted[_sweep_disp_cols].head(5).to_string(index=False))
print(f"\n  🏆 Best Policy: {best_score}  (LIFT_MIN = {LIFT_MIN:.2f}x, {'constrained' if _constrained else 'UNCONSTRAINED fallback'})")

print()
print("=" * 80)
print("B4 EVALUATION — BEST POLICY (full VAL)")
print("=" * 80)
print(eval_best_full.to_string(index=False))

print()
print("=" * 80)
print("IMPACTO ANUAL — inferencia de recuperación de ventas")
print(f"  (annual_factor = 52/{VAL_WEEKS} = {ANNUAL_FACTOR:.4f})")
print("=" * 80)
_all_v = eval_best_full[eval_best_full["season_group"] == "ALL"].iloc[0]
_sv    = eur_severity_tp_year
print(f"  eur_expected_year               : {eur_expected_year:>14,.2f}  €/año")
print(f"  eur_severity_tp_year (upper)    : {eur_severity_tp_year:>14,.2f}  €/año")
print(f"  eur_recuperable E=0.50 (baja)   : {eur_rec_E50_year:>14,.2f}  €/año")
print(f"  eur_recuperable E=0.70 (media)  : {eur_rec_E70_year:>14,.2f}  €/año")
print(f"  eur_recuperable E=0.90 (alta)   : {eur_rec_E90_year:>14,.2f}  €/año")

print()
print("=" * 80)
print("CORRECTION FACTORS (cf) PER SEASON")
print("=" * 80)
print(df_cf_factors[["season_group", "cf", "n_tune", "viol_before", "viol_after"]].to_string(index=False))

print()
print("=" * 80)
print("Archivos generados en NB_DIR / REPORT_DIR / OUT_DIR:")
print("=" * 80)
output_files = [
    ("forecast_h4_enriched.csv",                      NB_DIR),
    ("alerts_top100_h4_bestpolicy.csv",               OUT_DIR),
    ("eval_alerts_top100_h4_pooled_bestpolicy.csv",   REPORT_DIR),
    ("eval_coverage_h4_conditional.csv",              REPORT_DIR),
    ("eval_coverage_h4_conditional_v3.csv",           REPORT_DIR),
    ("b3_tuning_factors.csv",                         REPORT_DIR),
    ("policy_sweep_hybrid_results.csv",               REPORT_DIR),
    ("weekly_recovery_timeseries.csv",                REPORT_DIR),
    ("annual_impact_summary.md",                      REPORT_DIR),
    ("run_summary_h4.csv",                            REPORT_DIR),
]
for fname, base_dir in output_files:
    p = base_dir / fname
    if p.exists():
        print(f"  ✓ {fname}  ({p.stat().st_size:,} bytes)")
    else:
        print(f"  ✗ {fname}  [NO ENCONTRADO]")
print("=" * 80)